# 01 · Data quality

Runs the cleaning pipeline (`src/clean.py`), then inspects what it changed: the rule log, the quarantine table, the validation report and the first control totals.

Every number below is printed by code in this notebook.

In [1]:
import subprocess, sys
from pathlib import Path
import pandas as pd
pd.set_option("display.width", 160, "display.max_colwidth", 80)
ROOT = Path.cwd().parent
res = subprocess.run([sys.executable, str(ROOT / "src" / "clean.py")], capture_output=True, text=True)
print("pipeline exit code:", res.returncode)
CLEAN = ROOT / "data" / "clean"

pipeline exit code: 0


## Rule log

In [2]:
dq = pd.read_csv(CLEAN / "dq_log.csv")
dq

,rule_id,table,rule,rows_affected,action
0,R01,ledger,"Parse 3 mixed timestamp formats (ISO, dd/mm/yyyy, dd-Mon-yyyy 12h)",7265,standardised
1,R02,ledger,Strip thousands separators from text amounts,1165,standardised
2,R03,ledger,Blank payment mode (manual journal) set to 'Unknown',129,flagged
3,R04,ledger,Remove test postings (TEST- refs / C_TEST customer),40,quarantined
4,R05,ledger,"Remove duplicate postings (same order, type and amount; later voucher kept out)",72,quarantined
5,R06,ledger,Postings outside October IST,0,checked (none expected)
6,R07,ledger,"Rows in -> out (24,261 -> 24,149)",112,summary
7,R08,gateway,Remove exact duplicate export rows,48,quarantined
8,R09,gateway,"Normalise merchant_ref (case, spaces, '_' and missing '-')",4778,standardised
9,R10,gateway,Convert UTC to IST (rows whose calendar date changes),1725,standardised


## Quarantine: rows removed, by reason

In [3]:
q = pd.read_parquet(CLEAN / "quarantine.parquet")
q.groupby(["source", "reason"]).size().rename("rows").reset_index()

,source,reason,rows
0,gateway,duplicate_export_row,48
1,ledger,duplicate_posting,72
2,ledger,test_row,40


In [4]:
# Duplicate ERP postings: what do they look like? (same order, amount, different voucher)
q[q.reason == "duplicate_posting"][["voucher_no", "posting_ts", "order_ref", "amount_inr"]].head()

,voucher_no,posting_ts,order_ref,amount_inr
40,RV/900000,2026-10-04 20:16:31,ORD-1003173,1092.0
41,RV/900001,2026-10-17 23:02:55,ORD-1013505,908.0
42,RV/900002,2026-10-26 12:45:24,ORD-1020026,921.0
43,RV/900003,30/10/2026 21:54,ORD-1023383,1675.0
44,RV/900004,2026-10-26 08:35:52,ORD-1019865,384.0


## Validation report

In [5]:
val = pd.read_csv(CLEAN / "validation_report.csv")
print(f"{val.passed.sum()}/{len(val)} checks passed")
val

19/19 checks passed


,check,passed,detail
0,ledger voucher_no unique,True,NaN
1,ledger order_id matches ORD-nnnnnnn,True,NaN
2,ledger postings all in October IST,True,NaN
3,"ledger receipts positive, credit notes negative",True,NaN
4,ledger one receipt per order,True,NaN
5,ledger no nulls in key columns,True,NaN
6,gateway txn_id unique,True,NaN
7,gateway order_id matches ORD-nnnnnnn,True,NaN
8,gateway method mapped,True,NaN
9,gateway fee between 0% and 3% of amount,True,NaN


## First control totals

Before any matching, do the four sources tie at the total level? (They should not — that is the point of the exercise.)

In [6]:
led = pd.read_parquet(CLEAN / "ledger.parquet")
gw = pd.read_parquet(CLEAN / "gateway_txns.parquet")
po = pd.read_parquet(CLEAN / "payouts.parquet")
bank = pd.read_parquet(CLEAN / "bank.parquet")

oct_gw = gw[gw.in_period_ist]
totals = pd.Series({
    "ERP receipts": led.loc[led.entry_type == "receipt", "amount_inr"].sum(),
    "ERP credit notes": led.loc[led.entry_type == "credit_note", "amount_inr"].sum(),
    "Gateway captured (Oct IST, in UTC report)": oct_gw.loc[(oct_gw.type == "payment") & (oct_gw.status == "captured"), "amount_inr"].sum(),
    "Gateway refunds (Oct IST, in UTC report)": -oct_gw.loc[oct_gw.type == "refund", "amount_inr"].sum(),
    "Payouts net (settled in Oct)": po.loc[po.settlement_date.dt.month == 10, "net_inr"].sum(),
    "Bank PayFlux credits": bank.loc[bank.category == "gateway_settlement", "credit"].sum(),
}).round(2)
totals.map("₹{:,.2f}".format)

ERP receipts                                 ₹22,215,188.00
ERP credit notes                               ₹-223,956.50
Gateway captured (Oct IST, in UTC report)    ₹22,040,966.00
Gateway refunds (Oct IST, in UTC report)       ₹-329,031.50
Payouts net (settled in Oct)                 ₹20,449,800.31
Bank PayFlux credits                         ₹20,449,800.31
dtype: str

In [7]:
erp_net = totals["ERP receipts"] + totals["ERP credit notes"]
gw_net = totals["Gateway captured (Oct IST, in UTC report)"] + totals["Gateway refunds (Oct IST, in UTC report)"]
print(f"ERP net sales                 ₹{erp_net:,.2f}")
print(f"Gateway net (before fees)     ₹{gw_net:,.2f}")
print(f"Bank PayFlux credits          ₹{totals['Bank PayFlux credits']:,.2f}")
print(f"ERP net - bank                ₹{erp_net - totals['Bank PayFlux credits']:,.2f}")

ERP net sales                 ₹21,991,231.50
Gateway net (before fees)     ₹21,711,934.50
Bank PayFlux credits          ₹20,449,800.31
ERP net - bank                ₹1,541,431.19


**Reading:** the totals don't tie, and the gap mixes several causes — fees and GST, refunds not booked, timing (settlements in transit at month end and the IST/UTC cut-off) and genuine breaks. Totals alone cannot separate them; that needs transaction-level matching (notebook 02 onward).